In [ ]:
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()
openai_client = OpenAI()

In [ ]:
from gitsource import GithubRepositoryDataReader

reader = GithubRepositoryDataReader(
    repo_owner="DataTalksClub",
    repo_name="llm-zoomcamp",
    commit_id="8c1834d",
    allowed_extensions={"md"},
    filename_filter=lambda path: "/lessons/" in path,
)

files = reader.read()

In [ ]:
documents = []

for file in files:
    doc = file.parse()
    documents.append(doc)

# Q01 How many lesson pages are in the dataset?

In [ ]:
len(documents)

# Q02. What's the filename of the first result?

In [ ]:
question='How does the agentic loop keep calling the model until it stops?'

In [ ]:
from minsearch import Index

index = Index(
    text_fields=['content'],
    keyword_fields=["filename"]
)

index.fit(documents)

In [ ]:
index.search(question)[0]['filename']

# Q03. Use gpt-5.4-mini. How many input (prompt) tokens did we send to the model for this request?

In [ ]:
from hw_helper import RAGBase

In [ ]:
question = "How does the agentic loop keep calling the model until it stops?"

In [ ]:
assistant = RAGBase(index,openai_client)

In [ ]:
ans= assistant.rag("How does the agentic loop keep calling the model until it stops?")

In [ ]:
ans

input prompt tokens sent are 7126

# Q04. How many chunks do you get?

In [ ]:
from gitsource import chunk_documents

chunks = chunk_documents(documents, size=2000, step=1000)

In [ ]:
len(chunks)

# Q05. Compare the input tokens with Q3. How many fewer input tokens does the chunked version send?

In [ ]:
# for documents
index = Index(
    text_fields=['content'],
    keyword_fields=["filename"]
)

index.fit(documents)

# for chunks
index_chunks = Index(
    text_fields=['content'],
    keyword_fields=["filename"]
)

index_chunks.fit(chunks)

In [ ]:
# for documents
assistant = RAGBase(index,openai_client)

# for chunks
assistant_chunks = RAGBase(index_chunks,openai_client)

In [ ]:
answer_doc = assistant.rag("How does the agentic loop keep calling the model until it stops?")
answer_chunks = assistant_chunks.rag("How does the agentic loop keep calling the model until it stops?")

In [ ]:
answer_doc['input_tokens']

In [ ]:
final_answer = answer_doc['input_tokens']/answer_chunks['input_tokens']
print(final_answer)

# 3 * fewer input tokens does the chunked version send.

# Q06 How many times did the agent call search?

In [ ]:
question = 'How does the agentic loop work, and how is it different from plain RAG?'

In [ ]:
instructions = """
You're a course teaching assistant.
You're given a question from a course student and your task is to answer it.

Answer the student's question using the search tool. 
Make multiple searches with different keywords before answering.

If you want to look up information, use the search function. 
Use as many keywords from the user question as possible when making first requests.

Make multiple searches. First perform search, analyze the results 
and then perform more searches. 

The question has to be about the course or its logistics, offtopic questions 
shouldn't be answered. If the search returns nothing, it's likely an off-topic question.
If you can't answer the question using FAQ, don't do it yourself. Only use the 
facts from the FAQ database.

At the end, ask if there are other areas that the user wants to explore.
""".strip()

In [ ]:
index_chunks = Index(
    text_fields=['content'],
    keyword_fields=["filename"]
)

index_chunks.fit(chunks)

In [ ]:
def rag(self, query):
    search_results = self.search(query)
    prompt = self.build_prompt(query, search_results)
    answer = self.llm(prompt)
    return answer

In [ ]:
messages = [
    {"role": "user", "content": "I just discovered the course. Can I join it?"}
]

response = openai_client.responses.create(
    model="gpt-5.4-mini",
    input=messages,
    tools=[search_tool],
)

response.output_text

In [ ]:
# letting the agent to use the search tool to answer questions about the course

def search(query: str) -> dict[str, str]:
    """
    Search the FAQ database for entries matching the given query.
    """
    return index_chunks.search(
        query,
        num_results=5,
    )

In [ ]:
search_tool = {
    "type": "function",
    "name": "search",
    "description": "Search the FAQ database for entries matching the given query.",
    "parameters": {
        "type": "object",
        "properties": {
            "query": {
                "type": "string",
                "description": "Search query text to look up in the course FAQ."
            }
        },
        "required": ["query"],
        "additionalProperties": False
    }
}

In [ ]:
response = openai_client.responses.create(
    model="gpt-5.4-mini",
    input=messages,
    tools=[search_tool],
)

response.output

In [ ]:
import json

call = response.output[0]
args = json.loads(call.arguments)

results = search(**args)
result_json = json.dumps(results, indent=2)

In [ ]:
# sending the result back to the model
messages.extend(response.output)

messages.append({
    "type": "function_call_output",
    "call_id": call.call_id,
    "output": result_json,
})

In [ ]:
# Asking the model again
response = openai_client.responses.create(
    model="gpt-5.4-mini",
    input=messages,
    tools=[search_tool],
)

response.output_text

In [ ]:
# a function call helper
def make_call(call):
    args = json.loads(call.arguments)

    if call.name == "search":
        result = search(**args)

    result_json = json.dumps(result, indent=2)

    return {
        "type": "function_call_output",
        "call_id": call.call_id,
        "output": result_json,
    }

In [ ]:
# processing the question with the agentic loop

question = "I just discovered the course. Can I join it?"

messages = [
    {"role": "developer", "content": instructions},
    {"role": "user", "content": question},
]

response = openai_client.responses.create(
    model="gpt-5.4-mini",
    input=messages,
    tools=[search_tool],
)

messages.extend(response.output)
has_function_calls = False

for item in response.output:
    if item.type == "function_call":
        print("function_call:", item.name, item.arguments)
        call_output = make_call(item)
        messages.append(call_output)
        has_function_calls = True

    elif item.type == "message":
        print("ASSISTANT:")
        print(item.content[0].text)

In [ ]:
def agent_loop(instructions, question, model="gpt-5.4-mini") -> str:
    messages = [
        {"role": "developer", "content": instructions},
        {"role": "user", "content": question}
    ]

    it = 1

    while True:
        print(f"iteration #{it}...")
        has_function_calls = False

        response = openai_client.responses.create(
            model=model,
            input=messages,
            tools=[search_tool]
        )

        messages.extend(response.output)

        for item in response.output:
            if item.type == "function_call":
                print("function_call:", item.name, item.arguments)
                call_output = make_call(item)
                messages.append(call_output)
                has_function_calls = True

            elif item.type == "message":
                print("ASSISTANT:")
                last_answer = item.content[0].text
                print(item.content[0].text)

        it = it + 1
        if has_function_calls == False:
            break

    return last_answer

In [ ]:
agent_loop(instructions, "How does the agentic loop work, and how is it different from plain RAG")

#search has been call 3 times